# State 관리, MessagesState 와 Reducer
- 챗봇·에이전트는 매 턴 메시지가 입니다. 매번 `state["messages"] = [...]` 로 통째 덮어쓰면 이전 메시지가 사라지죠. 
- LangGraph 의 **Reducer (병합 함수)** 가 "이전 값 + 새 값" 을 어떻게 합칠지 결정합니다.

## 환경 준비


```
OPENAI_API_KEY=sk-...

```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")

## 1. 문제, 단순 덮어쓰기는 메시지를 잃는다

## 2. 해결, `add_messages` reducer

- State 필드에 `Annotated[list, add_messages]` 로 reducer 를 붙이면 LangGraph 가 **이전 메시지 + 새 메시지** 를 자동으로 이어붙임. 같은 ID 메시지는 자동 dedup.

### `Annotated[list, add_messages]` 가 하는 일

- `Annotated` 는 "타입 + 메타데이터" 를 동시에 적는 파이썬 문법. 여기서 메타데이터 자리에 들어간 `add_messages` 가 reducer.

- 노드가 `{"messages": [새 메시지]}` 를 반환하면
- LangGraph 가 **기존 messages 리스트와 새 리스트를 합쳐서** 새 state 를 만듦
- reducer 없으면 단순 덮어쓰기 (1번 예시 처럼 이전 메시지가 사라짐)

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]     


def chatbot_node(state: ChatState) -> dict:
    ai = llm.invoke(state["messages"])
    return {"messages": [ai]}               

In [ ]:
result = 

print("최종 messages 길이:", len(result["messages"]))
for m in result["messages"]:
    print(f"  [{type(m).__name__}] {m.content[:60]}")

## 3. `MessagesState`, 자주 쓰는 패턴 한 줄 단축

- `messages` 필드를 자주 쓰니까 LangGraph 가 **`MessagesState`** 라는 미리 만든 클래스를 제공. 위 ChatState 와 동일.

## 4. 멀티턴, 같은 그래프를 반복 호출

- 각 호출 결과를 다음 호출의 입력으로 넘기면 대화 누적.

### 4.1. 대화 저장하고 이어하기, `InMemorySaver` + `thread_id`

- `global state` 변수 트릭은 한 노트북 안에서만 됩니다. 진짜 챗봇은 **사용자별 / 세션별로 대화를 따로 저장** 해야 하죠. LangGraph 의 **checkpointer** 가 이걸 자동으로 해줍니다.

- `compile(checkpointer=...)` 로 그래프에 저장소 붙임
- 호출 시 `config={"configurable": {"thread_id": "user-123"}}` 로 세션 ID 지정
- 같은 thread_id 로 다시 호출하면 이전 대화가 자동으로 복원

### 4.2. `app.stream` 으로 노드별 진행 보기

- `invoke` 는 최종 결과만 돌려주지만, `stream` 으로 받으면 **노드 하나 끝날 때마다 중간 결과**가 흘러옵니다. 디버깅·UI 진행바에 활용.

In [ ]:
for chunk in app_saved.stream(
    {"messages": [HumanMessage("  ")]},
    config=,
    stream_mode=,
):
    for node_name, payload in chunk.items():
        last = payload["messages"][-1]
        print(f"[{node_name}] {type(last).__name__}: {last.content[:60]}")

In [ ]:
state = {"messages": []}

def chat(user_text):
    global state
    state["messages"].append(HumanMessage(user_text))
    state = app.invoke(state)         
    return state["messages"][-1].content


print("Q1:", ".")
print("A1:", chat(""))
print()
print("Q2:", "")
print("A2:", chat(""))  

## 5. (심화) 커스텀 State + 여러 필드 reducer

- `messages` 외에 다른 필드도 reducer 를 지정합니다. 예: 점수 누적.

In [ ]:
from typing import Annotated
from operator import add        


class QuizState(TypedDict):
    messages: Annotated[list, add_messages]
    score: Annotated[int, add]          


def grade_node(state: QuizState) -> dict:
    """마지막 답이 '예' 면 점수 +1."""
    last = state["messages"][-1].content.strip()
    return {"score": 1 if "예" in last else 0}


graph = StateGraph(QuizState)
graph.add_node("grade", grade_node)
graph.add_edge(START, "grade")
graph.add_edge("grade", END)
app_quiz = graph.compile()

## 6. (심화) Pydantic State

- TypedDict 대신 Pydantic 모델도 가능. 필드 검증·기본값 지원.

In [ ]:


graph = StateGraph(PydState)
graph.add_node("count", turn_counter)
graph.add_edge(START, "count")
graph.add_edge("count", END)
app_p = graph.compile()

state = 
state = 
print("1회 후 turn:", state["turn"])
state = 
print("2회 후 turn:", state["turn"])

## 7. 정리

| 패턴 | 언제 |
|---|---|
| `TypedDict + Annotated[list, add_messages]` | 채팅·에이전트 표준 |
| `MessagesState` | 위와 동일한 한 줄 단축 |
| `Annotated[int, add]` 같은 다른 reducer | 점수·카운트·합산 필드 |
| `Pydantic BaseModel` | 필드 검증·기본값 필요할 때 |


## [실습]

1. `MessagesState` 로 3턴 대화 (내 이름 -> 문의한 주문번호 -> 해당 문의 응대 방향) 가 모두 기억되는지 확인.
2. `chatbot_node` 의 system 프롬프트를 "고객지원 매니저 톤으로 답해" 로 바꿔보기.
3. `QuizState` 의 `score` 외에 `correct_questions: Annotated[list, add]` 필드 추가해서 맞춘 질문 누적.
4. Pydantic State 에 `remaining_sla_minutes: int = 100` 추가하고 매 노드에서 -10 씩 감소시키는 reducer 만들기 (`lambda old, new: old + new` 직접).
5. `messages` 가 50개 넘으면 자동으로 오래된 거 잘라내는 wrapper 노드 추가.
